In [ ]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# Load data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
series = data['Passengers']

# Step 1: Train-test split (Last 10 points for testing)
train = series.iloc[:-10]
test = series.iloc[-10:]

# Step 2: Perform ADF test on training data
result = adfuller(train)

# Extract and print the p-value
print(f"p-value: {result[1]:.2f}")

p-value: 0.99


In [ ]:
import pandas as pd

# Load data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
series = data['Passengers']

# Step 1: Train-test split
train = series.iloc[:-10]

# Step 3: Apply first-order differencing
differenced_train = train.diff().dropna()

# Calculate the average (mean)
average_diff = differenced_train.mean()

print(f"Average of differenced training data: {average_diff:.2f}")

Average of differenced training data: 2.10


In [ ]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# Load and Split
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
train = data['Passengers'].iloc[:-10]

# Apply first-order differencing
differenced_train = train.diff().dropna()

# Re-test stationarity
result_diff = adfuller(differenced_train)

# Output the p-value
print(f"p-value after differencing: {result_diff[1]:.2f}")

p-value after differencing: 0.07


In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Load and Split
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
train = data['Passengers'].iloc[:-10]

# Apply first-order differencing (as done in previous steps)
differenced_train = train.diff().dropna()

# Step 4: Build ARIMA model with (p,d,q) = (1,0,1)
# We fit this on the differenced data
model = ARIMA(differenced_train, order=(1, 0, 1))
model_fit = model.fit()

# Output the AIC score
print(f"AIC Score: {model_fit.aic:.2f}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


AIC Score: 1275.95


In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# 1. Setup and Split
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, index_col='Month', parse_dates=True)
train = data['Passengers'].iloc[:-10]
test = data['Passengers'].iloc[-10:]

# 2. Difference the training data
differenced_train = train.diff().dropna()

# 3. Fit ARIMA(1,0,1) on the differenced data
model = ARIMA(differenced_train, order=(1, 0, 1))
model_fit = model.fit()

# 4. Forecast 10 steps (this gives you predicted DIFFERENCES)
diff_forecast = model_fit.forecast(steps=10)

# 5. Reverse the differencing to get actual Passenger counts
# We take the last known value from the original training set
# and cumulatively add the forecasted differences.
last_val = train.iloc[-1]
forecasted_values = last_val + diff_forecast.cumsum()

# 6. Calculate MSE
mse = mean_squared_error(test, forecasted_values)

print(f"Mean Squared Error: {round(mse)}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Mean Squared Error: 12159
